In [ ]:
!pip install -q optuna

In [ ]:
import random
import time
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ----------------------------- Reproducibility -----------------------------
SEED = 42

def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)   # seeds python, numpy and TensorFlow

set_seeds()

# ------------------------------- Configuration -----------------------------
TARGET_COL      = "WS10M"
TRAIN_FRAC      = 0.80     # chronological train share (used if TEST_START_DATE is None)
TEST_START_DATE = None     # e.g. "2024-01-01" to hold out everything from that date onward

N_SPLITS        = 5        # TimeSeriesSplit folds (inside the training period only)
INNER_VAL_FRAC  = 0.15     # share of training windows used ONLY for early stopping
MAX_EPOCHS      = 100
BATCH_SIZE      = 32
PATIENCE_CV     = 5
PATIENCE_FINAL  = 10

# Grid search space
PARAM_GRID = {
    "window_size":   [12, 24],
    "n_filters":     [64, 128],
    "kernel_size":   [3, 5],
    "n_layers":      [1, 2],
    "dense_units":   [64, 128],
    "dropout_rate":  [0.2, 0.3],
    "learning_rate": [1e-3, 3e-3],
    "activation":    ["relu", "tanh"],
}

# Random search space
RANDOM_SPACE = {
    "window_size":   list(range(3, 25)),
    "n_filters":     [32, 64, 128, 256],
    "kernel_size":   list(range(2, 6)),
    "n_layers":      list(range(1, 4)),
    "dense_units":   [32, 64, 128, 256],
    "dropout_rate":  [0.1, 0.2, 0.3, 0.4, 0.5],
    "learning_rate": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "activation":    ["relu", "tanh", "elu", "sigmoid"],
}
N_TRIALS_RANDOM = 50
N_TRIALS_OPTUNA = 50
OPTUNA_TIMEOUT  = 3600     # seconds

## 1. Load data and create the chronological train / test split

In [ ]:
sheet_id   = "1j_Euo80PrGckVDVr2hTG9zZebxJD0TSC"
sheet_name = "Sheet1"
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df = pd.read_csv(csv_url)

# Guarantee chronological order before anything else (time-series split assumes it)
if {"YEAR", "MO", "DY"}.issubset(df.columns):
    df = df.sort_values(["YEAR", "MO", "DY"]).reset_index(drop=True)

df = df.set_index("Date")
df = df.drop(columns=[c for c in ["YEAR", "MO", "DY"] if c in df.columns])

series = df[TARGET_COL].to_numpy(dtype="float64")

# Basic sanity checks
assert not np.isnan(series).any(), "Missing values found in the target — handle them before modelling."
if series.min() <= -900:
    warnings.warn("Sentinel values (e.g. -999) detected in the target — clean them before modelling.")

# ---- ONE chronological split; everything else derives from it ----
n_obs = len(series)
if TEST_START_DATE is None:
    split_idx = int(n_obs * TRAIN_FRAC)
else:
    split_idx = int(np.searchsorted(pd.to_datetime(df.index), pd.Timestamp(TEST_START_DATE)))

series_train = series[:split_idx]           # the ONLY data hyper-parameter tuning may touch
y_test_true  = series[split_idx:]           # test targets (raw units) — used for reporting only
naive_pred   = series[split_idx - 1:-1]     # persistence benchmark: y_t = y_{t-1}

print(f"Total observations : {n_obs}")
print(f"Training period    : {split_idx} obs ({df.index[0]} → {df.index[split_idx - 1]})")
print(f"Test period        : {n_obs - split_idx} obs ({df.index[split_idx]} → {df.index[-1]})")

## 2. Leakage-safe utilities

In [ ]:
def make_windows(scaled, window):
    # Sliding windows: window i uses scaled[i:i+window] to predict scaled[i+window]
    n = len(scaled) - window
    X = np.lib.stride_tricks.sliding_window_view(scaled, window)[:n]
    y = scaled[window:]
    return X[..., None].astype("float32"), y.astype("float32")


def resolve_params(params):
    # Kernel cannot exceed window_size - 1. Applied identically in search and final training.
    p = dict(params)
    p["window_size"] = int(p["window_size"])
    p["kernel_size"] = int(min(p["kernel_size"], p["window_size"] - 1))
    return p


def build_cnn(p):
    w, k = p["window_size"], p["kernel_size"]
    model = Sequential()
    model.add(Input(shape=(w, 1)))
    length = w
    for _ in range(int(p["n_layers"])):
        if length < k:                       # no room left for another convolution
            break
        model.add(Conv1D(int(p["n_filters"]), k, activation=p["activation"]))
        length = length - k + 1
        if length >= 2:
            model.add(MaxPooling1D(pool_size=2))
            length //= 2
    model.add(Flatten())
    model.add(Dense(int(p["dense_units"]), activation=p["activation"]))
    model.add(Dropout(float(p["dropout_rate"])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=float(p["learning_rate"])), loss="mse")
    return model


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mse = mean_squared_error(y_true, y_pred)
    nz = np.abs(y_true) > 1e-8                                   # avoid division by zero in MAPE
    mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100 if nz.any() else np.nan
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y_true, y_pred),
            "MAPE": mape, "R2": r2_score(y_true, y_pred)}


def print_metrics(m, title):
    print(f"\n{title}\n" + "-" * 40)
    for k, v in m.items():
        print(f"{k:>6}: {v:.4f}" + ("%" if k == "MAPE" else ""))
    print("-" * 40)


def cv_mse(params):
    # Expanding-window CV on the TRAINING PERIOD ONLY. In every fold:
    #   * the scaler is fitted on that fold's training observations only,
    #   * early stopping uses the last INNER_VAL_FRAC of the fold's training windows,
    #   * the fold's validation block is used only for scoring (in original units).
    p = resolve_params(params)
    w = p["window_size"]
    n_windows = len(series_train) - w
    scores = []

    for tr_idx, va_idx in TimeSeriesSplit(n_splits=N_SPLITS).split(np.arange(n_windows)):
        last_train_target = tr_idx[-1] + w                       # raw index of the last training target
        scaler = MinMaxScaler().fit(series_train[: last_train_target + 1].reshape(-1, 1))
        scaled = scaler.transform(series_train.reshape(-1, 1)).ravel()
        X, y = make_windows(scaled, w)

        n_es = max(1, int(len(tr_idx) * INNER_VAL_FRAC))
        fit_idx, es_idx = tr_idx[:-n_es], tr_idx[-n_es:]

        set_seeds()
        model = build_cnn(p)
        model.fit(X[fit_idx], y[fit_idx],
                  validation_data=(X[es_idx], y[es_idx]),
                  epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=0,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE_CV,
                                           restore_best_weights=True)])

        pred = scaler.inverse_transform(model.predict(X[va_idx], verbose=0)).ravel()
        true = series_train[w:][va_idx]                          # raw targets of the validation block
        scores.append(mean_squared_error(true, pred))
        tf.keras.backend.clear_session()

    return float(np.mean(scores))


def train_and_evaluate(params, method, cv_score=None):
    # Final fit on the training period, single evaluation on the test period.
    p = resolve_params(params)
    w = p["window_size"]

    # Scaler sees the training period only
    scaler = MinMaxScaler().fit(series_train.reshape(-1, 1))
    scaled = scaler.transform(series.reshape(-1, 1)).ravel()
    X, y = make_windows(scaled, w)

    n_train_win = split_idx - w                                  # windows whose target lies in the training period
    tr_idx = np.arange(0, n_train_win)
    te_idx = np.arange(n_train_win, len(y))                      # windows whose target lies in the test period
    assert len(te_idx) == len(y_test_true)

    n_es = max(1, int(len(tr_idx) * INNER_VAL_FRAC))
    fit_idx, es_idx = tr_idx[:-n_es], tr_idx[-n_es:]

    set_seeds()
    model = build_cnn(p)
    history = model.fit(X[fit_idx], y[fit_idx],
                        validation_data=(X[es_idx], y[es_idx]),      # early stopping on TRAIN-period data
                        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, verbose=1,
                        callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE_FINAL,
                                                 restore_best_weights=True)])

    inv = lambda a: scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()
    train_pred = inv(model.predict(X[tr_idx], verbose=0))
    test_pred  = inv(model.predict(X[te_idx], verbose=0))
    y_train_true = series[w:split_idx]

    train_m = regression_metrics(y_train_true, train_pred)
    test_m  = regression_metrics(y_test_true, test_pred)
    print_metrics(train_m, f"{method} — Training metrics")
    print_metrics(test_m,  f"{method} — Test metrics")

    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    ax[0].plot(y_test_true, label="Actual", color="blue", alpha=0.7)
    ax[0].plot(test_pred, label="Predicted", color="red", alpha=0.7)
    ax[0].set(title=f"{method}: Wind speed prediction (test set)\nRMSE {test_m['RMSE']:.3f}, MAE {test_m['MAE']:.3f}",
              xlabel="Time steps (days)", ylabel=f"{TARGET_COL} (m/s)")
    ax[0].legend(); ax[0].grid(True)
    ax[1].plot(history.history["loss"], label="Train loss")
    ax[1].plot(history.history["val_loss"], label="Early-stopping loss (train period)")
    ax[1].set(title=f"{method}: training history", xlabel="Epoch", ylabel="MSE (scaled)")
    ax[1].legend(); ax[1].grid(True)
    plt.tight_layout(); plt.show()

    return {"Method": method, "CV_MSE": cv_score, "Window": w, **{f"Test_{k}": v for k, v in test_m.items()},
            "Train_RMSE": train_m["RMSE"], "Params": p}


def show_best(params, score):
    print("\nBest parameters found")
    print("-" * 40)
    for k, v in resolve_params(params).items():
        print(f"  {k:15}: {v:.5f}" if isinstance(v, float) else f"  {k:15}: {v}")
    print(f"  {'CV MSE':15}: {score:.5f}")
    print("-" * 40)


# Benchmark: persistence forecast on the same test targets
naive_m = regression_metrics(y_test_true, naive_pred)
print_metrics(naive_m, "Persistence benchmark (y_t = y_{t-1}) — Test metrics")

summary = []

## 3. Grid Search (time-series CV on the training period only)

In [ ]:
def grid_search():
    keys = list(PARAM_GRID.keys())
    combos = list(product(*PARAM_GRID.values()))
    print(f"Total parameter combinations: {len(combos)}")
    records, best = [], (None, float("inf"))
    t0 = time.time()

    for i, combo in enumerate(combos, 1):
        params = resolve_params(dict(zip(keys, combo)))
        try:
            score = cv_mse(params)
        except Exception as e:
            print(f"[{i}/{len(combos)}] failed: {e}")
            continue
        records.append({**params, "cv_mse": score})
        if score < best[1]:
            best = (params, score)
            print(f"[{i}/{len(combos)}] new best CV MSE {score:.5f}  {params}")
    print(f"Grid search finished in {(time.time() - t0) / 60:.1f} min")
    return best[0], best[1], pd.DataFrame(records).sort_values("cv_mse").reset_index(drop=True)


grid_best_params, grid_best_cv, grid_results = grid_search()
show_best(grid_best_params, grid_best_cv)
grid_results.head(10)

In [ ]:
summary.append(train_and_evaluate(grid_best_params, "Grid Search", grid_best_cv))

## 4. Random Search (time-series CV on the training period only)

In [ ]:
def random_search(n_trials=N_TRIALS_RANDOM):
    rng = random.Random(SEED)
    records, best = [], (None, float("inf"))
    t0 = time.time()

    for trial in range(1, n_trials + 1):
        params = resolve_params({k: rng.choice(v) for k, v in RANDOM_SPACE.items()})
        try:
            score = cv_mse(params)
        except Exception as e:
            print(f"[{trial}/{n_trials}] failed: {e}")
            continue
        records.append({**params, "cv_mse": score})
        if score < best[1]:
            best = (params, score)
            print(f"[{trial}/{n_trials}] new best CV MSE {score:.5f}  {params}")
    print(f"Random search finished in {(time.time() - t0) / 60:.1f} min")
    return best[0], best[1], pd.DataFrame(records).sort_values("cv_mse").reset_index(drop=True)


random_best_params, random_best_cv, random_results = random_search()
show_best(random_best_params, random_best_cv)
random_results.head(10)

In [ ]:
summary.append(train_and_evaluate(random_best_params, "Random Search", random_best_cv))

## 5. Optuna / TPE (time-series CV on the training period only)

In [ ]:
def objective(trial):
    params = {
        "window_size":   trial.suggest_int("window_size", 3, 24),
        "n_filters":     trial.suggest_int("n_filters", 32, 256),
        "kernel_size":   trial.suggest_int("kernel_size", 2, 5),     # clamped to window_size-1 in resolve_params
        "n_layers":      trial.suggest_int("n_layers", 1, 3),
        "dense_units":   trial.suggest_int("dense_units", 32, 256),
        "dropout_rate":  trial.suggest_float("dropout_rate", 0.1, 0.5),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "activation":    trial.suggest_categorical("activation", ["relu", "tanh", "elu"]),
    }
    return cv_mse(params)


study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, timeout=OPTUNA_TIMEOUT, catch=(Exception,))

optuna_best_params, optuna_best_cv = study.best_params, study.best_value
show_best(optuna_best_params, optuna_best_cv)

In [ ]:
summary.append(train_and_evaluate(optuna_best_params, "Optuna", optuna_best_cv))

## 6. Final comparison



In [ ]:
summary_df = pd.DataFrame(summary).drop(columns="Params")
benchmark_row = {"Method": "Persistence (benchmark)", "CV_MSE": np.nan, "Window": 1,
                 **{f"Test_{k}": v for k, v in naive_m.items()}}
summary_df = pd.concat([summary_df, pd.DataFrame([benchmark_row])], ignore_index=True)
summary_df.round(4)